# ML Network Intrusion Detector
**IBM SkillsBuild Cyber Security BootCamp 2026 (BharatCares) - Final Project**

**Goal:** Build a Scikit-learn model that classifies network connections as **normal** or **attack (anomaly)** using the public **NSL-KDD** dataset.

**Pipeline:** Load data -> Explore -> Preprocess -> Train 3 models -> Evaluate -> Feature importance -> Save model -> Conclusion

**Dataset:** NSL-KDD (University of New Brunswick) - https://www.unb.ca/cic/datasets/nsl.html
(mirror used for download: https://github.com/defcom17/NSL_KDD)

## 1. Import Libraries

In [ ]:
import os
import urllib.request
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("Libraries imported successfully")

## 2. Load the NSL-KDD Dataset
The files are downloaded automatically on first run. If you are offline, download `KDDTrain+.txt` and `KDDTest+.txt` manually from the dataset link above and place them in a `data/` folder next to this notebook.

In [ ]:
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

BASE = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
FILES = {"train": "KDDTrain+.txt", "test": "KDDTest+.txt"}

for name, fname in FILES.items():
    path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(path):
        print(f"Downloading {fname} ...")
        urllib.request.urlretrieve(BASE + fname, path)
    else:
        print(f"{fname} already present")

columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land",
    "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "label", "difficulty",
]

train_df = pd.read_csv(os.path.join(DATA_DIR, FILES["train"]), names=columns)
test_df = pd.read_csv(os.path.join(DATA_DIR, FILES["test"]), names=columns)
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
train_df.head()

## 3. Exploratory Data Analysis

In [ ]:
print("Missing values (train):", train_df.isnull().sum().sum())
print("Duplicate rows (train):", train_df.duplicated().sum())
train_df.drop(columns="difficulty").describe().T.head(12)

In [ ]:
# Binary target: 0 = normal, 1 = attack
for df in (train_df, test_df):
    df["target"] = (df["label"] != "normal").astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
train_df["target"].map({0: "Normal", 1: "Attack"}).value_counts().plot(
    kind="bar", ax=axes[0], color=["#2a9d8f", "#e76f51"])
axes[0].set_title("Class distribution (train)")
axes[0].set_ylabel("Connections")
axes[0].tick_params(axis="x", rotation=0)

train_df[train_df["target"] == 1]["label"].value_counts().head(10).plot(
    kind="barh", ax=axes[1], color="#e76f51")
axes[1].set_title("Top 10 attack types (train)")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["protocol_type", "flag", "service"]):
    order = train_df[col].value_counts().head(8).index
    sns.countplot(data=train_df[train_df[col].isin(order)], y=col, hue="target",
                  order=order, ax=ax, palette=["#2a9d8f", "#e76f51"])
    ax.set_title(f"{col} vs attack (0=normal, 1=attack)")
plt.tight_layout()
plt.show()

## 4. Preprocessing
- Drop the `label` and `difficulty` columns from the features
- One-hot encode the categorical columns: `protocol_type`, `service`, `flag`
- Align train and test columns (the test set may contain categories not seen in training)
- Scale numeric features with `StandardScaler`

In [ ]:
categorical = ["protocol_type", "service", "flag"]

X_train = pd.get_dummies(train_df.drop(columns=["label", "difficulty", "target"]), columns=categorical)
X_test = pd.get_dummies(test_df.drop(columns=["label", "difficulty", "target"]), columns=categorical)

# Align test columns to training columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df["target"].values
y_test = test_df["target"].values

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Feature count:", X_train.shape[1])
print("X_train:", X_train_s.shape, "| X_test:", X_test_s.shape)

## 5. Train Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(max_depth=15, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE),
}

results, predictions, probabilities = [], {}, {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    proba = model.predict_proba(X_test_s)[:, 1]
    predictions[name], probabilities[name] = pred, proba
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1-Score": f1_score(y_test, pred),
    })
    print(f"Trained {name}")

results_df = pd.DataFrame(results).set_index("Model").round(4)
results_df

## 6. Evaluation

In [ ]:
results_df.plot(kind="bar", figsize=(10, 4.5), ylim=(0.5, 1.0), colormap="viridis")
plt.title("Model comparison on NSL-KDD test set")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
best_name = results_df["F1-Score"].idxmax()
print("Best model by F1-score:", best_name, "\n")
print(classification_report(y_test, predictions[best_name], target_names=["Normal", "Attack"]))

In [ ]:
plt.figure(figsize=(7, 5))
for name, proba in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc(fpr, tpr):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Feature Importance (Random Forest)

In [ ]:
rf = models["Random Forest"]
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
importances.head(15).sort_values().plot(kind="barh", color="#264653")
plt.title("Top 15 features driving intrusion detection")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()
importances.head(10)

## 8. Save the Model and Predict on New Traffic

In [ ]:
os.makedirs("model", exist_ok=True)
best_model = models[best_name]
joblib.dump({"model": best_model, "scaler": scaler, "columns": list(X_train.columns)},
            "model/intrusion_detector.joblib")
print("Saved model to model/intrusion_detector.joblib")

# Demo: classify 5 random test connections
bundle = joblib.load("model/intrusion_detector.joblib")
sample = X_test.sample(5, random_state=RANDOM_STATE)
sample_pred = bundle["model"].predict(bundle["scaler"].transform(sample[bundle["columns"]]))
demo = pd.DataFrame({
    "Predicted": np.where(sample_pred == 1, "ATTACK", "Normal"),
    "Actual": np.where(y_test[sample.index] == 1, "ATTACK", "Normal"),
}, index=sample.index)
demo

## 9. Conclusion
- Three classifiers (Logistic Regression, Decision Tree, Random Forest) were trained on NSL-KDD to separate normal traffic from attacks.
- Tree-based models usually outperform the linear baseline because intrusion patterns are non-linear.
- Features such as `src_bytes`, `dst_bytes`, `service`, `same_srv_rate` and `count` are among the most informative.
- NSL-KDD's test set contains attack types absent from training, so test accuracy is lower than training accuracy. This is expected and mirrors real-world novel attacks.

**Limitations and future work:** try multi-class attack-type detection (DoS, Probe, R2L, U2R), handle class imbalance, tune hyperparameters with GridSearchCV, test on newer datasets (CICIDS2017, UNSW-NB15), and explore deep learning models.